# ComfyUI Fast Colab

Fast mode for mobile: starts plain ComfyUI without ComfyUI-Manager to reduce startup and blank-page hangs. Models/output persist in Google Drive under `ComfyUI_persist`.


## 1. Check GPU

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=False)


## 2. Fast local setup + Drive persistence


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

USE_GOOGLE_DRIVE = True
LOCAL_WORKSPACE = Path('/content/ComfyUI')
DRIVE_ROOT = Path('/content/drive/MyDrive/ComfyUI_persist')
UPSTREAM = 'https://github.com/comfyanonymous/ComfyUI.git'


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


def symlink_dir(target, link):
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.exists():
        if link.is_symlink():
            link.unlink()
        elif link.is_dir():
            shutil.rmtree(link)
        else:
            link.unlink()
    link.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(target), str(link))
    print('Linked:', link, '->', target)


if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Keep ComfyUI code on Colab local disk. Loading UI files from Drive is much slower.
if LOCAL_WORKSPACE.exists():
    shutil.rmtree(LOCAL_WORKSPACE)
run(['git', 'clone', '--depth', '1', UPSTREAM, str(LOCAL_WORKSPACE)])

if USE_GOOGLE_DRIVE:
    symlink_dir(DRIVE_ROOT / 'models', LOCAL_WORKSPACE / 'models')
    symlink_dir(DRIVE_ROOT / 'output', LOCAL_WORKSPACE / 'output')
    symlink_dir(DRIVE_ROOT / 'input', LOCAL_WORKSPACE / 'input')
    symlink_dir(DRIVE_ROOT / 'user', LOCAL_WORKSPACE / 'user')

run(['python3', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run(['python3', '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121'])
run(['python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=LOCAL_WORKSPACE)
run([
    'python3', '-m', 'pip', 'install', '-q',
    'alembic', 'blake3', 'GitPython', 'accelerate', 'einops', 'transformers',
    'safetensors', 'aiohttp', 'pyyaml', 'Pillow', 'scipy', 'tqdm', 'psutil',
    'tokenizers', 'torchsde', 'kornia', 'spandrel', 'soundfile', 'sentencepiece'
])

print('Fast local ComfyUI setup complete:', LOCAL_WORKSPACE)
print('Persistent files are stored in:', DRIVE_ROOT)


## 3. Skip ComfyUI-Manager for fast startup


In [ ]:
from pathlib import Path

WORKSPACE = Path('/content/ComfyUI')
print('Fast mode: ComfyUI-Manager is skipped.')
print('If you need custom nodes later, use the full notebook instead.')


## 4. Download starter models

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/ComfyUI')
(WORKSPACE / 'models' / 'checkpoints').mkdir(parents=True, exist_ok=True)
(WORKSPACE / 'models' / 'vae').mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


run([
    'wget', '-c',
    'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt',
    '-P', str(WORKSPACE / 'models' / 'checkpoints')
])
run([
    'wget', '-c',
    'https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors',
    '-P', str(WORKSPACE / 'models' / 'vae')
])


## 5. Fast launch

Starts ComfyUI and Cloudflare Tunnel as soon as `/system_stats` is ready. Use the `READY public URL`.


In [ ]:
from pathlib import Path
from google.colab import output
import re
import socket
import subprocess
import time
import urllib.request

WORKSPACE = Path('/content/ComfyUI')
PORT = 8188


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


def wait_for_port(port, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            return
        time.sleep(0.5)
    raise TimeoutError(f'Port {port} did not open')


def fetch_url(url, timeout=10):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return response.status


def wait_for_system_stats(port, timeout=180):
    print('Waiting for ComfyUI /system_stats...')
    wait_for_port(port, timeout=timeout)
    start = time.time()
    last_error = None
    while time.time() - start < timeout:
        try:
            if fetch_url(f'http://127.0.0.1:{port}/system_stats') == 200:
                print('ComfyUI backend is ready.')
                return
        except Exception as exc:
            last_error = exc
        time.sleep(1)
    raise TimeoutError(f'ComfyUI backend did not become ready: {last_error}')


def start_cloudflared(port):
    print('Starting Cloudflare Tunnel...')
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
    start = time.time()
    while time.time() - start < 120:
        line = proc.stderr.readline()
        if not line:
            time.sleep(0.2)
            continue
        match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if match:
            return proc, match.group(0)
    raise RuntimeError('Cloudflare Tunnel URL was not printed in time')


run(['wget', '-q', '-O', '/content/cloudflared-linux-amd64.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared-linux-amd64.deb'], check=False)

print('Starting ComfyUI server...')
comfy_proc = subprocess.Popen(
    ['python3', 'main.py', '--dont-print-server'],
    cwd=str(WORKSPACE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

wait_for_system_stats(PORT)
print('Showing local iframe in Colab...')
output.serve_kernel_port_as_iframe(PORT, height=900)
cloudflared_proc, public_url = start_cloudflared(PORT)
print()
print('READY public URL:', public_url)
print('If the page is blank on mobile, open it in Chrome external browser and refresh once after 10 seconds.')
print('Keep this cell running. Stop it only when you want to shut down ComfyUI.')

for line in comfy_proc.stdout:
    print(line, end='')
